# 7. Comunidades

Se reconstruyen la red bipartita autor-video y sus dos proyecciones (secciones 4 y 5) para detectar comunidades de participación.

In [1]:
from itertools import combinations
import os
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from networkx.algorithms import bipartite
from networkx.algorithms.community import louvain_communities, modularity

DATA_DIR = Path("data") if Path("data").exists() else Path("../data")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)

comments = pd.read_csv(DATA_DIR / "comments_procesado.csv", encoding="utf-8-sig")
videos = pd.read_csv(DATA_DIR / "videos_procesado.csv", encoding="utf-8-sig")

autores_ids = list(comments["author_channel_id"].unique())
videos_ids = list(comments["video_id"].unique())
titulo = videos.set_index("video_id")["title"].to_dict()
canal = videos.set_index("video_id")["channel_name"].to_dict()
categoria = videos.set_index("video_id")["category"].to_dict()
nombre_autor = comments.drop_duplicates("author_channel_id").set_index("author_channel_id")["author_name"]

# Se reconstruye la red bipartita de la sección 4 (misma definición de arista y peso).
aristas = comments.groupby(["author_channel_id", "video_id"]).size().reset_index(name="peso")

B = nx.Graph()
B.add_nodes_from(autores_ids, tipo="autor")
B.add_nodes_from(videos_ids, tipo="video")
for _, row in aristas.iterrows():
    B.add_edge(row["author_channel_id"], row["video_id"], weight=row["peso"])

# Proyecciones de la sección 5, reconstruidas de la misma forma.
AA = bipartite.weighted_projected_graph(B, autores_ids)
VV = bipartite.weighted_projected_graph(B, videos_ids)

print(f"Red bipartita: {B.number_of_nodes()} nodos, {B.number_of_edges()} aristas")
print(f"Proyección autor-autor: {AA.number_of_nodes()} nodos, {AA.number_of_edges()} aristas")
print(f"Proyección video-video: {VV.number_of_nodes()} nodos, {VV.number_of_edges()} aristas")

Red bipartita: 351 nodos, 343 aristas
Proyección autor-autor: 332 nodos, 10732 aristas
Proyección video-video: 19 nodos, 11 aristas


### 7.1 Selección de la red y justificación

La sección 6 estableció que la red bipartita se fragmenta en 10 componentes conexos, uno grande (81.5 % de los nodos, organizado alrededor de 10 de los 19 videos) y nueve componentes de un solo video con su propio público. Esa estructura, no la de las proyecciones, es la que interesa describir como "comunidades":

- **La proyección autor-autor no sirve para esto.** Cada video induce un cuasi-clique completo entre sus comentaristas (todos comparten el mismo vecino-video), de modo que cualquier algoritmo de comunidades sobre `AA` recuperaría casi exactamente los mismos grupos que ya están dados por la partición autor→video_principal, sin aportar nada nuevo: es un artefacto de la proyección, no un hallazgo.
- **La proyección video-video es demasiado tenue** (11 aristas entre 19 nodos, 9 aislados) para que la detección de comunidades sea informativa más allá de "hay una componente grande y nueve videos sueltos", que ya se reportó en 6.3.
- **La red bipartita conserva la unidad real de información** (qué autor comentó en qué video, con qué intensidad) sin la distorsión combinatoria de una proyección. Detectar comunidades directamente sobre `B` permite además que un mismo grupo incluya autores y videos a la vez, que es la pregunta sustantiva del laboratorio ("¿existen audiencias compartidas entre videos?").

Se usa entonces **la red bipartita `B`** para la detección de comunidades.

### 7.2 Algoritmo aplicado: Louvain

Se aplica el algoritmo de **Louvain** (`networkx.algorithms.community.louvain_communities`), que optimiza la modularidad por fusión greedy de nodos en pasadas sucesivas.

**Supuestos y tratamiento de pesos.**

- Louvain no exige que la red sea de un solo modo: opera sobre cualquier grafo ponderado no dirigido, incluida una red bipartita, tratando cada nodo (autor o video) como una unidad intercambiable del mismo grafo. El costo es que la modularidad estándar usa como referencia un modelo nulo de configuración que no distingue tipos de nodo, por lo que no impone que las comunidades alternen autor/video; en la práctica, dado que toda arista de `B` conecta un autor con un video, cada comunidad detectada contiene necesariamente ambos tipos.
- El peso de cada arista (`weight`, número de comentarios de ese autor en ese video) se usa directamente en la optimización: un autor con varios comentarios en el mismo video queda más "atraído" hacia la comunidad de ese video que uno con un solo comentario, aunque en esta red el 92 % de los pesos vale 1 (ver sección 2.7), por lo que el peso influye poco en el resultado.
- Se fija `seed=42` porque Louvain es no determinista (el orden de fusión afecta el resultado cuando hay empates); se reporta también la modularidad para verificar que la partición encontrada es informativa y no arbitraria.

In [2]:
comunidades = louvain_communities(B, weight='weight', seed=42)
comunidades = sorted(comunidades, key=len, reverse=True)
Q = modularity(B, comunidades, weight='weight')

id_a_comunidad = {}
for i, com in enumerate(comunidades):
    for nodo in com:
        id_a_comunidad[nodo] = i

tabla_com = []
for i, com in enumerate(comunidades):
    autores_c = [n for n in com if n in autores_ids]
    videos_c = [n for n in com if n in videos_ids]
    tabla_com.append({
        'comunidad': i,
        'n_nodos': len(com),
        'n_autores': len(autores_c),
        'n_videos': len(videos_c),
        'videos': ', '.join(titulo.get(v, v)[:35] for v in videos_c),
    })

tabla_com = pd.DataFrame(tabla_com)
print(f'Comunidades detectadas: {len(comunidades)}')
print(f'Modularidad: {Q:.4f}\n')
tabla_com

Comunidades detectadas: 17
Modularidad: 0.7774



,comunidad,n_nodos,n_autores,n_videos,videos
0,0,126,125,1,Qué rico come tu diputado
1,1,48,47,1,La cooptación de Walter Mazariegos
2,2,32,29,3,"Caminar en una ciudad hecha para ca, Internet:..."
3,3,31,30,1,Inician los trabajos de recuperació
4,4,26,25,1,Plan 2032 Ciudad de Guatemala
5,5,19,18,1,EE.UU. envía a mexicanos deportados
6,6,19,18,1,Conferencia de Prensa del Gobierno
7,7,14,13,1,Capturan a presuntos delincuentes d
8,8,8,7,1,Bloqueos en Guatemala este 31 de ag
9,9,8,7,1,Capturan a ladrón que había quedado


**Cómo leer el número de comunidades.** Antes de interpretar la tabla anterior conviene contrastar el número de comunidades con el número de componentes conexas de `B`: si coinciden uno a uno, Louvain no aporta más que lo que ya mostraba la sección 6 (la red es una colección de audiencias aisladas por video). Si en cambio una misma componente conexa se reparte en varias comunidades, eso indica subgrupos con audiencias más próximas entre sí que con el resto del mismo bloque conectado.

In [3]:
comp_b = sorted(nx.connected_components(B), key=len, reverse=True)
print(f'Componentes conexos de B: {len(comp_b)}')
print(f'Comunidades de Louvain: {len(comunidades)}')

# ¿Coincide cada comunidad con una única componente conexa?
comp_de_nodo = {}
for i, c in enumerate(comp_b):
    for n in c:
        comp_de_nodo[n] = i

coincide = all(
    len({comp_de_nodo[n] for n in com}) == 1
    for com in comunidades
)
print(f'Cada comunidad cae dentro de una sola componente conexa: {coincide}')

Componentes conexos de B: 10
Comunidades de Louvain: 17
Cada comunidad cae dentro de una sola componente conexa: True


**Resultado.** Cada una de las 17 comunidades queda dentro de una sola de las 10 componentes conexas (ninguna comunidad mezcla nodos de dos componentes distintas, como debe ser: no hay caminos entre componentes), pero **no hay una correspondencia uno a uno**: Louvain reparte la componente gigante (81.5 % de los nodos) en varias comunidades más pequeñas, mientras que las nueve componentes triviales de un solo video quedan intactas como comunidades propias. Es decir, dentro del bloque conectado sí existe subestructura adicional —grupos de videos cuyas audiencias se solapan más entre sí que con el resto del bloque— que la sola lectura de componentes conexas (sección 6) no distinguía. La modularidad alta (0.777) refleja ambos efectos a la vez: la separación entre componentes (la mayor parte) y esa subdivisión interna (una parte menor).

### 7.3 Número de comunidades, tamaños y modularidad

Ya reportados en la tabla de 7.2: el número de comunidades, el tamaño de cada una (nodos, autores, videos) y la modularidad `Q`. Una modularidad cercana a 1 en una red tan fragmentada no debe leerse como "estructura de comunidades muy rica", sino como confirmación de que casi toda la red se divide en bloques que apenas se tocan entre sí — la misma fragmentación ya vista en la sección 6, ahora cuantificada con una medida de calidad de partición.

### 7.4 Visualización de todas las comunidades

In [4]:
colores = plt.cm.tab20(np.linspace(0, 1, len(comunidades)))
color_de_nodo = {n: colores[id_a_comunidad[n]] for n in B.nodes()}

plt.figure(figsize=(13, 9))
pos = nx.spring_layout(B, k=0.18, iterations=70, seed=42)

nx.draw_networkx_edges(B, pos, alpha=0.15, edge_color="gray", width=0.6)
nx.draw_networkx_nodes(
    B, pos, nodelist=autores_ids, node_shape="o",
    node_size=25, node_color=[color_de_nodo[a] for a in autores_ids], alpha=0.85,
)
nx.draw_networkx_nodes(
    B, pos, nodelist=videos_ids, node_shape="s",
    node_size=120, node_color=[color_de_nodo[v] for v in videos_ids],
    edgecolors="black", linewidths=1.0,
)

# Etiquetas de las 5 comunidades principales (en el video con más comentarios de cada una)
for cid in range(min(5, len(comunidades))):
    vids_en_com = [n for n in comunidades[cid] if n in videos_ids]
    if vids_en_com:
        v_rep = max(vids_en_com, key=lambda v: B.degree(v))
        x, y = pos[v_rep]
        lbl = titulo.get(v_rep, v_rep)[:25] + "..."
        plt.text(x, y + 0.03, f"C{cid}: {lbl}", fontsize=8, fontweight="bold",
                 bbox=dict(boxstyle="round,pad=0.2", facecolor="white", alpha=0.8, edgecolor="none"))

plt.title(f"Detección de comunidades con Louvain sobre la red bipartita (17 comunidades, Q = {Q:.3f})\nCírculos = autores · Cuadrados con borde = videos · Colores = comunidad", fontsize=11)
plt.axis("off")
plt.tight_layout()
plt.savefig(DATA_DIR / "fig_comunidades.png", dpi=150, bbox_inches="tight")
plt.show()

<Figure size 1300x900 with 1 Axes>

### 7.5 Caracterización de las tres comunidades principales

Se caracteriza cada comunidad por sus videos, canal(es), número de autores, intensidad de participación (comentarios totales y promedio por autor) y palabras más frecuentes en `texto_limpio` como aproximación al tema. El sentimiento por comunidad se integra en la sección 9, una vez calculado el análisis de sentimiento sobre `texto_original`; aquí se deja preparada la columna `comunidad` en `comments` para ese cruce posterior.

In [5]:
comments['comunidad'] = comments['video_id'].map(id_a_comunidad)

from collections import Counter

def palabras_frecuentes(textos, top=8):
    contador = Counter()
    for t in textos.dropna():
        contador.update(str(t).split())
    return ', '.join(w for w, _ in contador.most_common(top))

resumen = []
for i in tabla_com.sort_values('n_nodos', ascending=False)['comunidad'].head(3):
    sub = comments[comments['comunidad'] == i]
    videos_c = [n for n in comunidades[i] if n in videos_ids]
    resumen.append({
        'comunidad': i,
        'n_videos': len(videos_c),
        'n_autores': sub['author_channel_id'].nunique(),
        'n_comentarios': len(sub),
        'comentarios_por_autor': round(len(sub) / max(sub['author_channel_id'].nunique(), 1), 2),
        'canales': ', '.join(sorted(set(canal.get(v, '?') for v in videos_c))),
        'categorias': ', '.join(sorted(set(categoria.get(v, '?') for v in videos_c))),
        'palabras_frecuentes': palabras_frecuentes(sub['texto_limpio']),
    })

resumen = pd.DataFrame(resumen)
resumen

,comunidad,n_videos,n_autores,n_comentarios,comentarios_por_autor,canales,categorias,palabras_frecuentes
0,0,1,128,161,1.26,Quorum,News & Politics,"pueblo, dinero, diputados, trabajo, diputado, ..."
1,1,1,49,50,1.02,Quorum,News & Politics,"usac, corruptos, universidad, estudiantes, exc..."
2,2,3,29,34,1.17,Quorum,News & Politics,"excelente, empresas, solo, internet, ley, país..."


**Lectura.** Cada una de las tres comunidades principales corresponde, en la práctica, a uno o pocos videos con su propio público de comentaristas y su propio vocabulario dominante; no se observan tres "conversaciones" temáticamente distintas que crucen canales, sino tres bloques de audiencia que coinciden con la fragmentación ya descrita en la sección 6. Esta lectura se completa en la sección 9 con el sentimiento dominante de cada bloque.

# 8. Nodos centrales y participantes puente

Se calculan medidas de centralidad sobre la red bipartita `B`, que es la que conserva la información original sin el artefacto de las proyecciones (sección 7.1).

### 8.1 Medidas de centralidad utilizadas

- **Grado (`degree`).** Para un autor, el grado es el número de videos distintos en los que comentó: mide *recurrencia* y, si se combina con el peso, *diversidad de participación*. Para un video, el grado es el número de autores distintos que comentaron en él: mide *alcance* de audiencia. Es la medida más directa y la única cuya interpretación no depende de que la red esté conectada.
- **Intermediación (`betweenness`).** Mide cuántos caminos más cortos entre pares de nodos pasan por un nodo dado. En una red bipartita muy fragmentada como esta, solo tiene sentido calcularla dentro de cada componente conexa (entre componentes no existen caminos), por lo que se calcula sobre la componente gigante y se interpreta como capacidad de un autor o video de *conectar* subgrupos que, sin él, quedarían separados.
- **Cercanía (`closeness`)** se descarta como medida principal: con 10 componentes desconectadas, la cercanía estándar solo es comparable dentro de una misma componente y la mayoría de las componentes pequeñas son estrellas triviales (un video, sus autores) donde la cercanía no aporta más que el grado. Se reporta únicamente sobre la componente gigante, a modo de complemento.
- **PageRank** se calcula como medida adicional porque, a diferencia del grado simple, pondera la importancia de los vecinos: un autor que comentó en un video muy comentado recibe más peso que uno que comentó en un video con pocos comentaristas. Se calcula con los pesos de arista (`weight`) sobre toda la red, incluidas las componentes pequeñas, porque PageRank sí está definido en grafos desconectados.

In [6]:
gc_nodos = max(nx.connected_components(B), key=len)
GC = B.subgraph(gc_nodos).copy()

deg_b = dict(B.degree())
deg_w_b = dict(B.degree(weight="weight"))
pr_b = nx.pagerank(B, weight="weight")

betw_gc = nx.betweenness_centrality(GC, weight="weight", normalized=True)
closeness_gc = nx.closeness_centrality(GC)

centralidad = pd.DataFrame({
    "nodo_id": list(B.nodes()),
    "tipo": ["autor" if n in autores_ids else "video" for n in B.nodes()],
    "nombre": [nombre_autor.get(n, n) if n in autores_ids else titulo.get(n, n) for n in B.nodes()],
    "grado": [deg_b[n] for n in B.nodes()],
    "grado_ponderado": [deg_w_b[n] for n in B.nodes()],
    "pagerank": [pr_b[n] for n in B.nodes()],
    "betweenness_gc": [betw_gc.get(n, 0.0) for n in B.nodes()],
    "closeness_gc": [closeness_gc.get(n, 0.0) for n in B.nodes()],
})

centralidad.to_csv(DATA_DIR / "centralidad.csv", index=False, encoding="utf-8-sig")
print(f"Guardado: {DATA_DIR}/centralidad.csv ({len(centralidad)} nodos)")

Guardado: ../data/centralidad.csv (351 nodos)


### 8.2 Interpretación separada para autores y videos

In [7]:
print('--- AUTORES: top 10 por grado (recurrencia) ---')
top_autores = centralidad[centralidad.tipo == 'autor'].sort_values('grado', ascending=False).head(10)
print(top_autores[['nombre', 'grado', 'grado_ponderado', 'pagerank', 'betweenness_gc']].to_string(index=False))

print('\n--- VIDEOS: top 10 por grado (alcance de audiencia) ---')
top_videos = centralidad[centralidad.tipo == 'video'].sort_values('grado', ascending=False).head(10)
print(top_videos[['nombre', 'grado', 'grado_ponderado', 'pagerank', 'betweenness_gc']].to_string(index=False))

--- AUTORES: top 10 por grado (recurrencia) ---
              nombre  grado  grado_ponderado  pagerank  betweenness_gc
      @inge_vergueta      3                3  0.003325        0.463652
       @hashojea7348      3                4  0.004536        0.041364
   @moisesvaldez4043      2                2  0.002477        0.048085
@franciscoflores3120      2                2  0.002294        0.087373
  @MarcosCarillo-b1r      2                2  0.002698        0.048085
 @virgiliogarcia3039      2                3  0.003039        0.349493
        @josegil3813      2                2  0.002078        0.266864
     @Alejandro00710      2                2  0.002454        0.097480
        @Jel.Awesh.M      2                3  0.003458        0.000000
   @Santiago.Nolasco      1                1  0.001867        0.000000

--- VIDEOS: top 10 por grado (alcance de audiencia) ---
                                                                                           nombre  grado  grado_po

**Autores.** El grado (número de videos distintos comentados) mide la *recurrencia*: como ya se estableció en 6.1, el 97.3 % de los autores tiene grado 1, así que cualquier autor con grado ≥ 2 destaca de inmediato en este ranking y es, por definición, el único tipo de autor capaz de tener `betweenness` positivo (conecta al menos dos videos). El grado ponderado añade la dimensión de *intensidad*: distingue a un autor que comentó una vez en dos videos distintos de otro que dejó varios comentarios en el mismo video. Ningún autor de esta muestra combina alta recurrencia con alta intensidad — la matriz de comentarios es demasiado dispersa para eso.

**Videos.** El grado mide *alcance*: cuántos autores distintos comentaron. El PageRank de un video sube además si sus comentaristas son, a su vez, autores que también comentaron en otros videos con audiencia propia — es decir, premia a los videos cuyo público se solapa con el de otros videos, no solo a los que tienen más comentarios en términos absolutos. La `betweenness` de un video identifica cuáles funcionan como puente entre dos grupos de autores que, sin ese video, no tendrían ningún autor en común.

### 8.3 Participantes recurrentes, autores puente y videos articuladores

Un **autor puente** es aquel cuya eliminación desconecta partes de la red que solo él conectaba. Se identifica de dos formas: (a) autores con grado > 1 (condición necesaria: solo ellos pueden tener aristas hacia más de un video) y (b) puntos de articulación de la componente gigante, que es la definición formal de "si lo quitáramos, la red se segmenta". Lo mismo se aplica a videos para identificar **videos articuladores**.

In [8]:
articulaciones = set(nx.articulation_points(GC))
autores_articulacion = [n for n in articulaciones if n in autores_ids]
videos_articulacion = [n for n in articulaciones if n in videos_ids]

print(f"Puntos de articulación en la componente gigante: {len(articulaciones)}")
print(f"  Autores articuladores: {len(autores_articulacion)}")
for a in autores_articulacion:
    print(f"    {nombre_autor[a]}  (grado {deg_b[a]}, videos: {[titulo.get(v, v)[:30] for v in B.neighbors(a)]})")

print(f"  Videos articuladores: {len(videos_articulacion)}")
for v in videos_articulacion:
    print(f"    {titulo.get(v, v)[:50]}  (grado {deg_b[v]})")


Puntos de articulación en la componente gigante: 17
  Autores articuladores: 7
    @josegil3813  (grado 2, videos: ['Inician los trabajos de recupe', 'Conferencia de Prensa del Gobi'])
    @hashojea7348  (grado 3, videos: ['Caminar en una ciudad hecha pa', 'Internet: escoger el menos mal', 'Arroz con pollo a la MONOPOLIO'])
    @moisesvaldez4043  (grado 2, videos: ['Bloqueos en Guatemala este 31 ', 'Qué rico come tu diputado'])
    @franciscoflores3120  (grado 2, videos: ['Capturan a presuntos delincuen', 'Inician los trabajos de recupe'])
    @inge_vergueta  (grado 3, videos: ['La cooptación de Walter Mazari', 'Qué rico come tu diputado', 'Internet: escoger el menos mal'])
    @MarcosCarillo-b1r  (grado 2, videos: ['Capturan a ladrón que había qu', 'La cooptación de Walter Mazari'])
    @virgiliogarcia3039  (grado 2, videos: ['Conferencia de Prensa del Gobi', 'Qué rico come tu diputado'])
  Videos articuladores: 10
    Conferencia de Prensa del Gobierno de Guatemala. #  (grado 19)
   

In [9]:
# Autores recurrentes: grado > 1 (condición necesaria para ser puente).
recurrentes = centralidad[(centralidad.tipo == 'autor') & (centralidad.grado > 1)]
print(f'Autores recurrentes (comentaron en más de un video): {len(recurrentes)} de {len(autores_ids)} '
      f'({len(recurrentes) / len(autores_ids):.1%})')
print(recurrentes[['nombre', 'grado', 'betweenness_gc']].sort_values('grado', ascending=False).to_string(index=False))

# Prueba directa: qué pasa con la componente gigante si se quitan los autores articuladores.
GC_sin = GC.copy()
GC_sin.remove_nodes_from(autores_articulacion)
comp_sin = sorted(nx.connected_components(GC_sin), key=len, reverse=True)
print(f'\nSin los {len(autores_articulacion)} autores articuladores, la componente gigante '
      f'({len(gc_nodos)} nodos) se divide en {len(comp_sin)} piezas '
      f'(la mayor conserva {len(comp_sin[0])} nodos).')

Autores recurrentes (comentaron en más de un video): 9 de 332 (2.7%)
              nombre  grado  betweenness_gc
       @hashojea7348      3        0.041364
      @inge_vergueta      3        0.463652
  @MarcosCarillo-b1r      2        0.048085
@franciscoflores3120      2        0.087373
        @josegil3813      2        0.266864
 @virgiliogarcia3039      2        0.349493
     @Alejandro00710      2        0.097480
   @moisesvaldez4043      2        0.048085
        @Jel.Awesh.M      2        0.000000

Sin los 7 autores articuladores, la componente gigante (286 nodos) se divide en 8 piezas (la mayor conserva 173 nodos).


**Hallazgos.** Los autores articuladores son, por construcción, un subconjunto de los autores recurrentes: no todo autor con grado > 1 es punto de articulación (puede conectar dos videos que ya estaban unidos por otro camino), pero todo punto de articulación de tipo autor tiene grado > 1. La prueba de remoción confirma que son estructuralmente frágiles: la componente gigante depende de un número pequeño de autores para mantenerse unida, y no existe redundancia de caminos alternativos entre los videos que esos autores conectan — coherente con la fragilidad ya descrita en la sección 6.2 (`AA_sin_puentes`), ahora localizada en autores y videos concretos en vez de en una cifra agregada.

Los videos articuladores, cuando existen, cumplen el rol simétrico: son el único punto de contacto entre dos grupos de autores. Ninguno de estos hallazgos debe leerse como que esos autores o videos "conectan comunidades" en un sentido social — solo significa que, dentro de los datos recolectados, ningún otro autor comentó en ambos videos a la vez.